6월 21일
: https://colab.research.google.com/drive/1dZ-4qfrg8yjdZpn68FDt4WXDoMBiVHXa?hl=ko#scrollTo=q-bdLdQrMJNU

# <mark> GPU 연결

In [ ]:
# 0. A100이 잘 연결됐다면 'cuda'가 출력됩니다.
# =====================================
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"현재 사용 중인 디바이스: {device}")

현재 사용 중인 디바이스: cuda


# <mark> 파일 불러오기 및 전처리

In [ ]:
# 1. 드라이브 연결
# =====================================
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# 아래 셀이 더 자세하다

# import os
# import zipfile

# # 💡 [필수 수정] 구글 드라이브 내에 파일들이 들어있는 폴더 경로를 적어주세요.
# DRIVE_DIR = "/content/drive/MyDrive/Colab Notebooks/과제/6주차/018.감성대화"

# # 코랩 가상환경(A100) 내부에 데이터를 저장할 경로
# LOCAL_DIR = "./emotion_data"
# os.makedirs(LOCAL_DIR, exist_ok=True)

# print("드라이브 내 파일 목록:", os.listdir(DRIVE_DIR))

In [ ]:
# 2. 파일 압축 풀기
# =====================================
import os
import zipfile
import shutil

# 1. 구글 드라이브 기본 경로 설정
BASE_DRIVE_DIR = "/content/drive/MyDrive/Colab Notebooks/과제/6주차/018.감성대화"

# 2. 실제 .part0 파일들이 들어있는 세부 폴더 경로 지정
TRAIN_DRIVE_DIR = os.path.join(BASE_DRIVE_DIR, "Training_221115_add/라벨링데이터")
VAL_DRIVE_DIR = os.path.join(BASE_DRIVE_DIR, "Validation_221115_add/라벨링데이터")

# 3. 코랩 가상환경(A100) 내부에 풀어서 저장할 경로
LOCAL_DIR = "./emotion_data"
os.makedirs(LOCAL_DIR, exist_ok=True)

# 압축 해제를 안전하게 수행하는 함수
def extract_ai_hub_data(drive_path, output_folder_name):
    if not os.path.exists(drive_path):
        print(f"❌ 경로를 찾을 수 없습니다: {drive_path}")
        return

    # 폴더 내에서 .part0 파일 찾기
    part0_file = None
    for f in os.listdir(drive_path):
        if f.endswith(".part0"):
            part0_file = f
            break

    if part0_file is None:
        print(f"❌ {output_folder_name} 폴더 내에 .part0 파일이 없습니다.")
        return

    src_file_path = os.path.join(drive_path, part0_file)
    tmp_zip_path = os.path.join(LOCAL_DIR, f"{output_folder_name}.zip")

    print(f"📦 드라이브에서 [{part0_file}] 가져오는 중... 잠시만 기다려주세요.")

    # 1단계: 가상환경으로 복사하면서 이름을 .zip으로 강제 변경
    shutil.copy(src_file_path, tmp_zip_path)

    # 2단계: 압축 해제
    target_extract_dir = os.path.join(LOCAL_DIR, output_folder_name)
    try:
        with zipfile.ZipFile(tmp_zip_path, 'r') as zip_ref:
            zip_ref.extractall(target_extract_dir)
        print(f"✅ {output_folder_name} 압축 해제 완료! -> 경로: {target_extract_dir}")
        print("📂 풀린 파일 목록:", os.listdir(target_extract_dir))
    except Exception as e:
        print(f"❌ 압축 해제 중 에러 발생: {e}")
    finally:
        # 가상환경 용량 관리를 위해 임시 zip 파일은 삭제
        if os.path.exists(tmp_zip_path):
            os.remove(tmp_zip_path)

# 🚀 실행 1: Training 데이터 압축 풀기
extract_ai_hub_data(TRAIN_DRIVE_DIR, "training_set")

print("-" * 50)

# 🚀 실행 2: Validation 데이터 압축 풀기
extract_ai_hub_data(VAL_DRIVE_DIR, "validation_set")

📦 드라이브에서 [감성대화말뭉치(최종데이터)_Training.zip.part0] 가져오는 중... 잠시만 기다려주세요.
✅ training_set 압축 해제 완료! -> 경로: ./emotion_data/training_set
📂 풀린 파일 목록: ['감성대화말뭉치(최종데이터)_Training.json']
--------------------------------------------------
📦 드라이브에서 [감성대화말뭉치(최종데이터)_Validation.zip.part0] 가져오는 중... 잠시만 기다려주세요.
✅ validation_set 압축 해제 완료! -> 경로: ./emotion_data/validation_set
📂 풀린 파일 목록: ['감성대화말뭉치(최종데이터)_Validation.json']


In [ ]:
# 3. 판다스로 로드하기
# =====================================
import pandas as pd

# 1. 압축이 풀려있는 폴더 경로 지정
TRAIN_DIR = "./emotion_data/training_set"
VAL_DIR = "./emotion_data/validation_set"

# 2. 폴더 내부에서 한글 깨짐과 상관없이 진짜 .json 파일 이름 자동으로 찾아내기
train_file = [f for f in os.listdir(TRAIN_DIR) if f.endswith('.json')][0]
val_file = [f for f in os.listdir(VAL_DIR) if f.endswith('.json')][0]

# 3. 발견한 진짜 이름으로 절대 틀릴 수 없는 경로 결합
TRAIN_JSON_PATH = os.path.join(TRAIN_DIR, train_file)
VAL_JSON_PATH = os.path.join(VAL_DIR, val_file)

# 데이터 로드 실행
try:
    df_train = pd.read_json(TRAIN_JSON_PATH)
    df_val = pd.read_json(VAL_JSON_PATH)

    print("📈 학습 데이터 크기(행, 열):", df_train.shape)
    print("📈 검증 데이터 크기(행, 열):", df_val.shape)

    print("\n📌 컬럼(열) 이름 목록:")
    print(df_train.columns.tolist())

    print("\n📌 학습 데이터 미리보기:")
    display(df_train.head(3))
except Exception as e:
    print(f"오류가 발생했습니다: {e}")

📈 학습 데이터 크기(행, 열): (51628, 2)
📈 검증 데이터 크기(행, 열): (6640, 2)

📌 컬럼(열) 이름 목록:
['profile', 'talk']

📌 학습 데이터 미리보기:


,profile,talk
0,"{'persona-id': 'Pro_05349', 'persona': {'perso...","{'id': {'profile-id': 'Pro_05349', 'talk-id': ..."
1,"{'persona-id': 'Pro_05349', 'persona': {'perso...","{'id': {'profile-id': 'Pro_05349', 'talk-id': ..."
2,"{'persona-id': 'Pro_05349', 'persona': {'perso...","{'id': {'profile-id': 'Pro_05349', 'talk-id': ..."


In [ ]:
# 4. 원본 파일 데이터 확인
# =====================================
import json

# (A) 원본에 파트가 몇 개인지
print("원본 파트들:", os.listdir(TRAIN_DRIVE_DIR))

# (C) 압축 푼 폴더에 json이 몇 개인지
jsons = [f for f in os.listdir(TRAIN_DIR) if f.endswith('.json')]
print("json 개수:", len(jsons), jsons[:3])

# (B) json '진짜 구조' 들여다보기
with open(TRAIN_JSON_PATH, encoding='utf-8') as f:
    raw = json.load(f)

# 첫 번째 데이터를 줄바꿈과 들여쓰기를 적용해 출력
if isinstance(raw, list):
    print(json.dumps(raw[0], indent=4, ensure_ascii=False))
else:
    print(json.dumps(list(raw.keys()), indent=4, ensure_ascii=False))

원본 파트들: ['감성대화말뭉치(최종데이터)_Training.zip.part0']
json 개수: 1 ['감성대화말뭉치(최종데이터)_Training.json']
{
    "profile": {
        "persona-id": "Pro_05349",
        "persona": {
            "persona-id": "A02_G02_C01",
            "human": [
                "A02",
                "G02"
            ],
            "computer": [
                "C01"
            ]
        },
        "emotion": {
            "emotion-id": "S06_D02_E18",
            "type": "E18",
            "situation": [
                "S06",
                "D02"
            ]
        }
    },
    "talk": {
        "id": {
            "profile-id": "Pro_05349",
            "talk-id": "Pro_05349_00053"
        },
        "content": {
            "HS01": "일은 왜 해도 해도 끝이 없을까? 화가 난다.",
            "SS01": "많이 힘드시겠어요. 주위에 의논할 상대가 있나요?",
            "HS02": "그냥 내가 해결하는 게 나아. 남들한테 부담 주고 싶지도 않고.",
            "SS02": "혼자 해결하기로 했군요. 혼자서 해결하기 힘들면 주위에 의논할 사람을 찾아보세요. ",
            "HS03": "",
            "SS03": ""
        }
 

In [ ]:
# 5. 원본 list 그대로 로드 (df 말고)
# [참고] https://aihub.or.kr/aihubdata/data/view.do?currMenu=115&topMenu=100&aihubDataSe=data&dataSetSn=86
# [참고2] https://github.com/jhwanflow/Korean-Sentiments-Classification
# 위 셀까지는 연령, 성별, 상황 등이 라벨링 되어있어서 구분이 되지 않았다.
# 원본을 참고하여 해당 과제에서 필요한 '감정'데이터만 불러왔다.
# 어노테이션 포맷 및 데이터 구조 > 설명서 및 활용가이드 다운로드 > 구축활용가이드 다운로드 에서 indexing 되어있던 감정의 원본을 확인
# =====================================
with open(TRAIN_JSON_PATH, encoding='utf-8') as f:
    train_data = json.load(f)

SEP = "[SEP]"

# 📌 가이드북 스크린샷 기반 60개 감정 코드 매핑 딕셔너리
EMO_MAP = {
    # 1. 분노 (E10 ~ E19)
    "E10": "분노", "E11": "툴툴대는", "E12": "좌절한", "E13": "짜증내는", "E14": "방어적인",
    "E15": "악의적인", "E16": "안달하는", "E17": "구역질 나는", "E18": "노여워하는", "E19": "성가신",

    # 2. 슬픔 (E20 ~ E29)
    "E20": "슬픔", "E21": "실망한", "E22": "비통한", "E23": "후회되는", "E24": "우울한", "E25": "마비된",
    "E26": "염세적인", "E27": "눈물이 나는", "E28": "낙담한", "E29": "환멸을 느끼는",

    # 3. 불안 (E30 ~ E39)
    "E30": "불안", "E31": "두려운", "E32": "스트레스 받는", "E33": "취약한", "E34": "혼란스러운(불안)",
    "E35": "당혹스러운", "E36": "회의적인", "E37": "걱정스러운", "E38": "조심스러운", "E39": "초조한",

    # 4. 상처 (E40 ~ E49)
    "E40": "상처", "E41": "질투하는", "E42": "배신당한", "E43": "고립된(상처)", "E44": "충격 받은",
    "E45": "불우한", "E46": "희생된", "E47": "억울한", "E48": "괴로워하는", "E49": "버려진",

    # 5. 당황 (E50 ~ E59)
    "E50": "당황", "E51": "고립된(당황)", "E52": "남의 시선을 의식하는", "E53": "외로운", "E54": "열등감",
    "E55": "죄책감", "E56": "부끄러운", "E57": "혐오스러운", "E58": "한심한", "E59": "혼란스러운(당황)",

    # 6. 기쁨 (E60 ~ E69)
    "E60": "기쁨", "E61": "감사하는", "E62": "사랑하는", "E63": "편안한", "E64": "만족스러운",
    "E65": "흥분되는", "E66": "느긋한", "E67": "안도하는", "E68": "신이 난", "E69": "자신하는"
}

def build_lines(data):
    lines = []
    for rec in data:
        # profile -> emotion -> type에서 감정 코드 추출 (예: E18)
        code = rec["profile"]["emotion"]["type"]

        # 💡 딕셔너리에 정의된 한글 단어로 변환 (없으면 코드 그대로 유지)
        emo = EMO_MAP.get(code, code)

        content = rec["talk"]["content"]
        for i in ["01", "02", "03"]:  # 최대 3턴 반복
            human  = content.get(f"HS{i}", "").strip()
            system = content.get(f"SS{i}", "").strip()

            if human and system:  # 대화 짝이 둘 다 존재할 때만 리스트에 추가
                lines.append(f"{human} {SEP} [{emo}] {system}")
    return lines

# 실행 및 데이터 확인
train_lines = build_lines(train_data)
print("총 줄 수:", len(train_lines))
print("\n📌 변환 완료 예시 (첫 3줄):")
for line in train_lines[:3]:
    print(line)

총 줄 수: 145948

📌 변환 완료 예시 (첫 3줄):
일은 왜 해도 해도 끝이 없을까? 화가 난다. [SEP] [노여워하는] 많이 힘드시겠어요. 주위에 의논할 상대가 있나요?
그냥 내가 해결하는 게 나아. 남들한테 부담 주고 싶지도 않고. [SEP] [노여워하는] 혼자 해결하기로 했군요. 혼자서 해결하기 힘들면 주위에 의논할 사람을 찾아보세요.
이번 달에 또 급여가 깎였어! 물가는 오르는데 월급만 자꾸 깎이니까 너무 화가 나. [SEP] [노여워하는] 급여가 줄어 속상하시겠어요. 월급이 줄어든 것을 어떻게 보완하실 건가요?


In [ ]:
codes = {rec["profile"]["emotion"]["type"] for rec in train_data}
print("데이터 등장 코드 수:", len(codes))     # 데이터가 실제로 쓰는 감정 종류
print("EMO_MAP 항목 수:", len(EMO_MAP))       # 지금은 59 → 60 되어야
print("매핑 안 된 코드:", codes - set(EMO_MAP.keys()))   # 비어있으면 정상

데이터 등장 코드 수: 60
EMO_MAP 항목 수: 60
매핑 안 된 코드: set()


In [ ]:
for rec in train_data[:5]:
    code = rec["profile"]["emotion"]["type"]
    human = rec["talk"]["content"].get("HS01", "")
    print(f"[{EMO_MAP.get(code, code)}] ({code})  ←  {human}")

[노여워하는] (E18)  ←  일은 왜 해도 해도 끝이 없을까? 화가 난다.
[노여워하는] (E18)  ←  이번 달에 또 급여가 깎였어! 물가는 오르는데 월급만 자꾸 깎이니까 너무 화가 나.
[노여워하는] (E18)  ←  회사에 신입이 들어왔는데 말투가 거슬려. 그런 애를 매일 봐야 한다고 생각하니까 스트레스 받아. 
[노여워하는] (E18)  ←  직장에서 막내라는 이유로 나에게만 온갖 심부름을 시켜. 일도 많은 데 정말 분하고 섭섭해.
[노여워하는] (E18)  ←  얼마 전 입사한 신입사원이 나를 무시하는 것 같아서 너무 화가 나.


In [ ]:
# 지금까지는 train data를 변환하고 확인했다.
# 이번 셀에서는 valudation data를 변환한다.
with open(VAL_JSON_PATH, encoding='utf-8') as f:
    val_data = json.load(f)
val_lines = build_lines(val_data)
print("검증 데이터 줄 수:", len(val_lines))

검증 데이터 줄 수: 17965


## <font color = 'red'> 정리 </font>

| 구분 |학습 데이터 (train) | 검증 데이터 (val) | 설명 |
| :--- | :--- | :--- |  :--- |
| JSON 원본 대화 수 | 51,628개 | 6,640개 | 한 세트의 대화(상황, 연령, 대화 3턴)가 통째로 담긴 파일 내부의 원본 개수입니다.
| 추출된 총 문장 줄 수 | 145,948개 | 17,965개 | 원본 대화 하나당 최대 3개의 문장 쌍(Human-System)을 한 줄씩 분리해서 새로 만든 최종 훈련용 데이터 개수입니다. |

# <mark> 토크나이저

In [ ]:
# 모든 줄을 줄바꿈으로 이어 하나의 긴 텍스트로 (줄바꿈 = 예시 경계 역할)
# 1) 학습용과 검증용 텍스트를 모두 합쳐서 전체 텍스트 생성
total_text = "\n".join(train_lines) + "\n" + "\n".join(val_lines)

# vocab: 등장하는 '문자 종류' 모으기 (중복 없음)
chars = sorted(set(total_text))
vocab_size = len(chars)

print("새로운 vocab_size:", vocab_size)
# print("문자 일부:", chars[:30])

from pprint import pprint

print("들어있는 문자 전부 출력")
pprint(chars[:], width=80, compact=True)

새로운 vocab_size: 1514
들어있는 문자 전부 출력
['\n', ' ', '!', "'", '(', ')', '.', '?', 'E', 'O', 'P', 'S', '[', ']', '~',
 '\xa0', '○', 'ㅇ', 'ㅠ', '가', '각', '간', '갇', '갈', '갉', '감', '갑', '값', '갓', '갔',
 '강', '갖', '같', '갚', '갛', '개', '객', '갠', '갯', '갰', '갱', '갸', '걀', '걔', '걘', '걜',
 '거', '걱', '건', '걷', '걸', '검', '겁', '것', '겄', '겅', '겆', '겉', '겋', '게', '겐', '겔',
 '겟', '겠', '겨', '격', '겪', '견', '결', '겸', '겹', '겼', '경', '곁', '계', '곈', '곗', '곘',
 '고', '곡', '곤', '곧', '골', '곪', '곬', '곯', '곰', '곱', '곳', '공', '과', '곽', '관', '괄',
 '괏', '광', '괘', '괜', '괴', '괸', '굉', '교', '굣', '구', '국', '군', '굳', '굴', '굵', '굶',
 '굼', '굽', '굿', '궁', '궂', '궈', '권', '궜', '궤', '귀', '귄', '귈', '귐', '귓', '규', '균',
 '귤', '그', '극', '근', '글', '긁', '금', '급', '긋', '긍', '긎', '기', '긴', '길', '김', '깁',
 '깃', '깅', '깊', '까', '깍', '깎', '깐', '깔', '깜', '깝', '깟', '깠', '깡', '깥', '깨', '깬',
 '깰', '깻', '깼', '꺼', '꺽', '꺾', '껀', '껄', '껌', '껍', '껏', '껐', '껑', '께', '껜', '껴',
 '꼈', '꼐', '꼬', '꼭', '꼰', '꼴', '꼼', '꼽', '꼿', '꽁', '꽂', '꽃', '꽉', '꽝', '꽤', '꾀',

In [ ]:
# 2) 문자 <-> 정수 매핑 (양방향)
stoi = {ch: i for i, ch in enumerate(chars)}   # 👉 stoi: 문자→정수
itos = {i: ch for i, ch in enumerate(chars)}   # 👉 itos: 정수→문자

# 3) 인코더 / 디코더
def encode(s):
    return [stoi[c] for c in s]                # 문자열 → 정수 리스트
def decode(l):
    return "".join(itos[i] for i in l)          # 블랭크2: 정수→문자, 어느 매핑을 써야 하지?

# 4) 왕복 검증 (encode 후 decode하면 원본과 같아야)
sample = train_lines[0]
print(sample, '\n')
print(encode(sample)[:20], '\n')
print(decode(encode(sample)) == sample, '\n')        # True면 정상

일은 왜 해도 해도 끝이 없을까? 화가 난다. [SEP] [노여워하는] 많이 힘드시겠어요. 주위에 의논할 상대가 있나요? 

[1035, 1023, 1, 988, 1, 1432, 352, 1, 1432, 352, 1, 215, 1031, 1, 945, 1024, 145, 7, 1, 1467] 

True 



## <font color = 'red'> 참고 </font>

| 변수 | 뜻 | 누가 정하나 | 예시 |
| :--- | :--- | :--- |  :--- |
| vocab_size | 문자 종류 수 | 데이터가 결정 | 수백~수천
|block_size | 한 번에 보는 문맥 길이(몇 글자 보고 다음 예측) | 우리가 정함 | 128 |
| batch_size | 한 번에 병렬 처리할 시퀀스 개수 | 우리가 정함 | 32|

In [ ]:
# 클로드가 짠 코드
# # 줄들을 이어붙여 정수 텐서로
# train_data_ids = torch.tensor(encode("\n".join(train_lines)), dtype=torch.long)
# val_data_ids   = torch.tensor(encode("\n".join(val_lines)),   dtype=torch.long)
# print("train ids:", train_data_ids.shape, train_data_ids.dtype)

# # 하이퍼파라미터 (우리가 정함)
# block_size = 128
# batch_size = 32

# def get_batch(split):
#     data = train_data_ids if split == "train" else val_data_ids
#     # 시작 위치를 batch_size개 무작위로 (끝을 넘지 않게)
#     ix = torch.randint(len(data) - block_size, (batch_size,))
#     x = torch.stack([data[i     : i+block_size]   for i in ix])   # 입력
#     y = torch.stack([data[i+1     : i+block_size+1] for i in ix])   # 출력: x를 '한 칸' 민 것
#     return x.to(device), y.to(device)

# xb, yb = get_batch("train")
# print("x:", xb.shape, " y:", yb.shape)

In [ ]:
# 💡 13번 셀 전체를 이 코드로 '통째로' 변경하고 실행해 주세요!

train_data_ids = torch.tensor(encode("\n".join(train_lines)), dtype=torch.long).to(device)
val_data_ids   = torch.tensor(encode("\n".join(val_lines)),   dtype=torch.long).to(device)

print("train ids:", train_data_ids.shape, train_data_ids.dtype)

block_size = 128
batch_size = 32

def get_batch(split):
    data = train_data_ids if split == "train" else val_data_ids

    # 1. 0~127 오프셋 벡터를 GPU에 생성
    arity = torch.arange(block_size, device=data.device)

    # 2. 32개의 랜덤 시작 인덱스를 GPU에 생성
    ix = torch.randint(len(data) - block_size, (batch_size,), device=data.device)

    # 3. 브로드캐스팅으로 (32, 128) 주소판 계산 (GPU 내부 연산)
    X_indices = ix[:, None] + arity

    # 4. 주소판을 가지고 한 번에 데이터 잘라내기
    x = data[X_indices]
    y = data[X_indices + 1]

    # 5. 최종 결과 반환
    return x, y

# 함수 작동 여부 테스트
xb, yb = get_batch("train")
print("x:", xb.shape, " y:", yb.shape)

train ids: torch.Size([11577366]) torch.int64
x: torch.Size([32, 128])  y: torch.Size([32, 128])


# <mark> 모델 본체
: 정수 시퀀스를 받아 임베딩 → self-attention(RoPE) → 다음 글자 logits를 내는 생성기

## <mark> 1) Head 1개

| 변수 | 뜻 | 값 | 독립? |
| :--- | :--- | :--- |  :--- |
| vocab_size | 문자 종류 수 | 데이터 결정 | o |
| n_embd | 토큰 1개를 표현하는 벡터 차원 (=c) | 384 | 우리가 정함
| n_head | 어텐션 헤드 개수 | 6 | 우리가 정함 |
| head_size | 헤드 1개가 맡는 차원 | n=embd//n_head=64 | x (n_embd와 n_head에 종속) |
| block_size | 문맥길이 (=t) | 128 | 우리가 정함|
| batch_size | 묶음 개수 (=b) | 32 | 우리가 정함|

In [ ]:
# 하이퍼파라미터
n_embd    = 384
n_head    = 6
head_size = n_embd // n_head   #64
n_layer   = 6
dropout   = 0.2

현재 모델에 입력된 텐서 $X$의 차원: 3차원 :

(Batch: 32, Time: 128, n_embd: 384)

|텐서 | shape | dim=-1 |
| :--- | :--- | :--- |
| emb(임베딩) | (B, T, C) = (32, 128, 384) | C = n_embd (벡터 성분/채널) |
| wei(어텐션 점수) | (B, T, T) = (32, 128, 128) | key 위치 (각 query가 주목할 후보) |

In [ ]:
import torch.nn as nn
from torch.nn import functional as F

def rope_cache(T, d, device):
    theta = 1.0 / (10000 ** (torch.arange(0, d, 2, device=device).float() / d))
    freqs = torch.outer(torch.arange(T, device=device).float(), theta)  # (T, d/2)
    emb = torch.cat([freqs, freqs], dim=-1)                              # (T, d)
    return emb.cos(), emb.sin()

def rotate_half(x):
    d = x.shape[-1]
    x1, x2 = x[..., :d//2], x[..., d//2:]
    return torch.cat([-x2, x1], dim=-1)

def apply_rope(x, cos, sin):     # x:(...,T,d), cos/sin:(T,d) → 브로드캐스트
    return x * cos + rotate_half(x) * sin

| 단계 | 코드 | shape |
| :--- | :--- | :--- |
| 입력 | x | (32, 128, 384) |
| k, q, v | Linear(384->64) | (32, 128, 64) |
| RoPE | apply_rope(q/k) | (32, 128, 64) : 모양 그대로, 값만 회전 |
| 점수 | q @ k.transpose | (32, 128, 128) : 위치 x 위치 |
| 마스크 + softmax | future 가리고 합1 | (32, 128, 128) |
| 출력 | wei @ v | (32, 128, 64) |

In [ ]:
class Head(nn.Module):
    """어텐션 헤드 1개(부품). 입력: 임베딩된 (B,T,C) float 벡터."""
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)   #입력 차원은 전체 임베딩 차원인 n_embd이고, 출력 차원은 이 어텐션 헤드가 담당할 크기인 head_size.
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)

        cos, sin = rope_cache(T, k.shape[-1], x.device)
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        # 어텐션 점수 = q와 k의 유사도 (스케일링)
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))  # 미래 가리기
        wei = F.softmax(wei, dim=-1)

        v = self.value(x)
        out = wei @ v
        return out

## <font color = 'red'> 참고 </font>

컴퓨터에서 3차원 상자(Tensor)를 다룰 때, 각 차원(축)
- dim=0 (첫 번째 축) : 가장 바깥쪽 축인 Batch(묶음) 기준

- dim=1 (두 번째 축) : 중간 축인 Time(질문 문장의 단어들) 기준

- dim=-1 또는 dim=2 (마지막 축) : 가장 안쪽 축인 Time(답변 후보 단어들) 기준

In [ ]:
tok_emb = nn.Embedding(vocab_size, n_embd).to(device)

xb, yb = get_batch("train")
emb = tok_emb(xb)                       # (32,128) 정수 → (32,128,384) 벡터
h = Head(head_size).to(device)
print(emb.shape, "->", h(emb).shape)    # (32,128,384) -> (32,128,64) 나오면 정상

torch.Size([32, 128, 384]) -> torch.Size([32, 128, 64])


## <mark> 2) Head 를 MultiHead로

In [ ]:
class MultiHeadAttention(nn.Module):
    """헤드 여러 개를 병렬로 돌리고 합치는 부품."""
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj  = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

## <mark> 3) Feedforward

In [ ]:
class FeedForward(nn.Module):
    """토큰마다 독립으로 적용되는 작은 신경망."""
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),   # 384 → 1536 (확장)
            nn.ReLU(),                       # 비선형
            nn.Linear(4 * n_embd, n_embd),   # 1536 → 384 (축소)
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)

## <mark> 4) Block — 어텐션 + FF, 각각 잔차 연결

In [ ]:
class Block(nn.Module):
    """트랜스포머 층 1개 = 멀티헤드 어텐션 + FF (+ 잔차 + LayerNorm)."""
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa   = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1  = nn.LayerNorm(n_embd)
        self.ln2  = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))     # 잔차: 원본 x에 '어텐션 결과'를 더함
        x = x + self.ffwd(self.ln2(x))
        return x

In [ ]:
xb, yb = get_batch("train")
emb = tok_emb(xb)                       # (32,128,384)
block = Block(n_embd, n_head).to(device)
print(emb.shape, "->", block(emb).shape)   # (32,128,384) -> (32,128,384) 유지돼야 정상

torch.Size([32, 128, 384]) -> torch.Size([32, 128, 384])


## <mark> 5) 전체 모델

| -- | Head 등 부품 | 전체 모델 |
| :--- | :--- | :--- |
| 입력 | float 벡터 (B, T, C) | 정수 토큰 (B, T) |
| 첫 일 | 바로 q,k,v | 먼저 임베딩 (정수 -> 벡터) |

In [ ]:
class GPTLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)   # 정수→벡터 (입구)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])  # Block 6개
        self.ln_f   = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)                    # 벡터→vocab 점수 (출구)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.token_embedding_table(idx)    # 임베딩 : (B,T)→(B,T,C)
        x = self.blocks(x)                     # (B,T,C)
        x = self.ln_f(x)
        logits = self.lm_head(x)               # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits  = logits.view(B*T, C)      # (B*T, vocab_size)
            targets = targets.view(B*T)        # (B*T,)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]    # 문맥은 최근 block_size개만
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]          # 다음 글자 위치
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, idx_next], dim=1)
        return idx

## <mark> 6) 만들고 + 초기 상태 점검

In [ ]:
model = GPTLanguageModel().to(device)
print(sum(p.numel() for p in model.parameters())/1e6, "M 파라미터", "\n")

xb, yb = get_batch("train")
logits, loss = model(xb, yb)
print(logits.shape, "| 초기 loss:", loss.item())
import math
print("기대 초기 loss ≈ ln(vocab_size) =", math.log(vocab_size))

11.804906 M 파라미터 

torch.Size([4096, 1514]) | 초기 loss: 7.477439880371094
기대 초기 loss ≈ ln(vocab_size) = 7.322510433997394


In [ ]:
# 학습을 1도 안한 상태 : 가중치가 전부 무작위
ctx = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(ctx, max_new_tokens=200)[0].tolist()))


더괏홱깼앤넙였E맘핍찐쟁묶낡속줏냇군번빌짙퀼모꼰궈쩡쐴썹솥뷰잴에꼰깡튈곡똘태걷쿠차론밍펀듣싶엇만근류혁꾸인추량쬐별있적컸치삐꼴빅이윷엽섯뻐명운앳팡춘뵌퍼뛰궂흔글뀔우델허털커숴해젤뚜처윷눈항꿎승훌석엑탈몰검관뚤름꼿놈흩넸흡됩할삼킴쟀츠래펌뺀렉풋퀼곰샘똘펐럭길벅튈킥호컬음쿵역쫓니찢렵꽁샹챙늉님없속철키껄뺨헉과뛰령족훤낀볼산롯짠줫슥곤개옥범끽뻔교 흘객혐창촉렉률남솟진컹웃컸괴짐같왈홱버적쑤함망쩌뤘햇덮쌩


## <mark> 7) 학습 루프

In [ ]:
max_iters     = 5000
eval_interval = 500
eval_iters    = 200
learning_rate = 3e-4

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()                          # 평가 모드(dropout 끔)
    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()                         # 다시 학습 모드
    return out

for it in range(max_iters):
    if it % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {it}: train {losses['train']:.3f}, val {losses['val']:.3f}")

    xb, yb = get_batch("train")
    logits, loss = model(xb, yb)              # 순전파: loss 계산
    optimizer.zero_grad(set_to_none=True)     # 1) 이전 기울기 비우기
    loss.backward()                           # 2) 역전파(기울기 계산)
    optimizer.step()                          # 3) 가중치 갱신

step 0: train 7.471, val 7.470
step 500: train 1.806, val 1.798
step 1000: train 1.654, val 1.649
step 1500: train 1.575, val 1.581
step 2000: train 1.525, val 1.510
step 2500: train 1.475, val 1.465
step 3000: train 1.442, val 1.444
step 3500: train 1.416, val 1.418
step 4000: train 1.398, val 1.403
step 4500: train 1.376, val 1.380


In [ ]:
# ============= 학습 결과 =============
# step 0: train 7.471, val 7.470
# step 500: train 1.806, val 1.798
# step 1000: train 1.654, val 1.649
# step 1500: train 1.575, val 1.581
# step 2000: train 1.525, val 1.510
# step 2500: train 1.475, val 1.465
# step 3000: train 1.442, val 1.444
# step 3500: train 1.416, val 1.418
# step 4000: train 1.398, val 1.403
# step 4500: train 1.376, val 1.380

## <mark> 8) 챗봇 함수

In [ ]:
def respond(user_text, max_new_tokens=200):
    # 학습 포맷의 '앞부분'만 깔아줌 → 감정+응답은 모델이 이어 생성
    prompt = f"{user_text} {SEP} "          # 사람문장 뒤, 응답을 유도하는 구분자
    idx = torch.tensor([encode(prompt)], dtype=torch.long, device=device)

    out = model.generate(idx, max_new_tokens=max_new_tokens)[0].tolist()
    text = decode(out)

    # 우리가 넣은 prompt 이후 ~ 첫 줄바꿈까지가 '한 응답'
    generated = text[len(prompt):].split("\n")[0]
    return generated

# 테스트
print(respond("오늘 발표를 망쳐서 하루 종일 우울했어."))
print(respond("드디어 시험이 끝나서 너무 신난다!"))
print(respond("친구가 약속을 또 어겨서 서운해."))

[우울한] 노력만큼 만나서 기분이 좋지 않으셨겠어요.
[신이 난] 이웃이라 기곳에 시험 이식을 받아서 기쁘시겠어요.
[슬픔] 최근에 오랜 개월여를 내라고 것이라는 말만 들어 화가 나시겠어요.


## <mark> 9) 학습한 모델 저장

In [ ]:
torch.save(model.state_dict(), "/content/drive/MyDrive/Colab Notebooks/과제/6주차/emotion_gpt.pt")
import pickle
with open("/content/drive/MyDrive/Colab Notebooks/과제/6주차/tokenizer.pkl", "wb") as f:
    pickle.dump({"stoi": stoi, "itos": itos}, f)
print("저장 완료")

저장 완료
